# Task 1: Web Scraping — Tech Job Market Data
**CodeAlpha Data Analytics Internship**

**Goal:** Collect job postings for 3 roles (Data Analyst, Software Engineer, Full Stack Developer)
from TimesJobs.com, extracting title, company, location, experience required, and key skills.

We use TimesJobs instead of Naukri/LinkedIn because it renders job listings as plain server-side
HTML (no login wall, no heavy JavaScript), which makes it reliable to scrape with `requests` +
`BeautifulSoup` within a short timeframe. Naukri and LinkedIn both actively block scrapers and
often require JS rendering (Selenium) or logins, which eats time you don't have.


In [10]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import certifi

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}

ROLES = ["Data Analyst", "Software Engineer", "Full Stack Developer"]
POSTINGS_PER_ROLE = 50   # ~150 total across 3 roles
BASE_URL = "https://www.timesjobs.com/candidate/job-search.html"


## Scraper function
One request per search page; each page returns ~25 listings, so we paginate until we hit our target.

In [11]:
def scrape_role(role_name, target_count=50):
    """Scrape job postings for one role from TimesJobs and return a list of dicts."""
    results = []
    page = 1

    while len(results) < target_count:
        params = {
            "searchType": "personalizedSearch",
            "from": "submit",
            "txtKeywords": role_name,
            "txtLocation": "",
            "sequence": page,
            "startPage": 1,
        }
        try:
            resp = requests.get(
                BASE_URL, 
                headers=HEADERS, 
                params=params, 
                timeout=10,
                verify=certifi.where()
            )

            resp.raise_for_status()
        except requests.RequestException as e:
            print(f"  Request failed on page {page} for '{role_name}': {e}")
            break

        soup = BeautifulSoup(resp.text, "lxml")
        job_cards = soup.find_all("li", class_="clearfix job-bx wht-shd-bx")

        if not job_cards:
            print(f"  No more listings found for '{role_name}' at page {page} (stopping).")
            break

        for card in job_cards:
            if len(results) >= target_count:
                break
            try:
                title = card.find("h2").get_text(strip=True) if card.find("h2") else None
                company = card.find("h3", class_="joblist-comp-name")
                company = company.get_text(strip=True) if company else None

                location_tag = card.find("ul", class_="top-jd-dtl clearfix")
                location = None
                experience = None
                if location_tag:
                    li_items = location_tag.find_all("li")
                    if len(li_items) > 0:
                        experience = li_items[0].get_text(strip=True)
                    if len(li_items) > 1:
                        location = li_items[1].get_text(strip=True)

                skills_tag = card.find("span", class_="srp-skills")
                skills = skills_tag.get_text(strip=True, separator=",") if skills_tag else None

                results.append({
                    "role_searched": role_name,
                    "job_title": title,
                    "company": company,
                    "location": location,
                    "experience": experience,
                    "skills": skills,
                })
            except Exception as e:
                print(f"  Skipped one malformed listing: {e}")
                continue

        print(f"  '{role_name}': collected {len(results)}/{target_count} so far (page {page})")
        page += 1
        time.sleep(random.uniform(1.5, 3.0))  # be polite, avoid getting blocked

    return results


## Run the scraper for all 3 roles

In [12]:
all_results = []

for role in ROLES:
    print(f"Scraping role: {role}")
    role_results = scrape_role(role, POSTINGS_PER_ROLE)
    all_results.extend(role_results)
    print(f"  -> Done. Total collected so far: {len(all_results)}\n")

df = pd.DataFrame(all_results)
print(f"\nTotal postings collected: {len(df)}")
df.head(10)


Scraping role: Data Analyst
  Request failed on page 1 for 'Data Analyst': HTTPSConnectionPool(host='www.timesjobs.com', port=443): Max retries exceeded with url: /candidate/job-search.html?searchType=personalizedSearch&from=submit&txtKeywords=Data+Analyst&txtLocation=&sequence=1&startPage=1 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)')))
  -> Done. Total collected so far: 0

Scraping role: Software Engineer
  Request failed on page 1 for 'Software Engineer': HTTPSConnectionPool(host='www.timesjobs.com', port=443): Max retries exceeded with url: /candidate/job-search.html?searchType=personalizedSearch&from=submit&txtKeywords=Software+Engineer&txtLocation=&sequence=1&startPage=1 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)')))
  -> Done. Total collected so

""


## ⚠️ If the scraper returns 0 or very few results

Websites change their HTML structure over time, and TimesJobs' class names (`job-bx`, `joblist-comp-name`,
etc.) may have shifted since this was written. If you get empty results:

1. Open https://www.timesjobs.com/candidate/job-search.html?txtKeywords=Data+Analyst in your browser.
2. Right-click a job listing → **Inspect** → find the actual `<li>`/`<h2>`/`<h3>` class names being used now.
3. Update the `find()` / `find_all()` calls above to match.

This is completely normal for web scraping and shows you actually understand *why* the scraper works,
which is worth mentioning in your LinkedIn video.

## Save to CSV

In [5]:
df.to_csv("jobs_data.csv", index=False)
print("Saved to jobs_data.csv")
df.info()


Saved to jobs_data.csv
<class 'pandas.DataFrame'>
RangeIndex: 0 entries
Empty DataFrame
